In [ ]:
import os
import json
import time
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from PIL import Image
import pandas as pd
import itertools
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from pytorch_grad_cam import HiResCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, roc_curve, average_precision_score, cohen_kappa_score)
from sklearn.calibration import calibration_curve

In [ ]:
class Config:
    # Paths, Update as per your local / HPC setup
    data_dir       = r"C:\Users\harimenshi\Documents\python\iris\data"
    chaksu_dir     = r"C:\Users\harimenshi\Documents\python\iris\data\chaksu"
    arcmia_dir     = r"C:\Users\harimenshi\Documents\python\iris\data\arcmia"
    orgia_dir      = r"C:\Users\harimenshi\Documents\python\iris\data\orgia"
    refuge_dir     = r"C:\Users\harimenshi\Documents\python\iris\data\refuge2"
    output_dir     = r"C:\Users\harimenshi\Documents\python\iris\outputs"

    checkpoint_dir = os.path.join(output_dir, "checkpoints")
    metrics_dir    = os.path.join(output_dir, "metrics")
    figures_dir    = os.path.join(output_dir, "figures")
    cam_dir        = os.path.join(figures_dir, "cam")
    reports_dir    = os.path.join(output_dir, "reports")

    # Training - Adjust these parameters as needed
    image_size     = 224
    batch_size     = 32
    num_epochs     = 30
    learning_rate  = 1e-4
    weight_decay   = 5e-4
    patience       = 10

    # Normalization
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]

    # Device subgroups for fairness evaluation, e.g., different camera devices
    chaksu_devices = ['Bosch', 'Forus', 'Remidio']

config = Config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
os.makedirs(config.checkpoint_dir, exist_ok=True)
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.metrics_dir, exist_ok=True)
os.makedirs(config.figures_dir, exist_ok=True)
os.makedirs(config.cam_dir, exist_ok=True)
os.makedirs(config.reports_dir, exist_ok=True)

datasets = {"chaksu": config.chaksu_dir, "arcmia": os.path.join(config.arcmia_dir, 'arcmia'), "orgia": os.path.join(config.orgia_dir, 'orgia'), "refuge": os.path.join(config.refuge_dir, 'refuge')}

In [ ]:
import os
print(os.listdir(config.data_dir))
print(os.listdir(config.chaksu_dir))

In [ ]:
img_clahe_pil = Image.fromarray(np.array(Image.open(os.path.join(config.chaksu_dir, 'Bosch', 'train', '0', 'image_0001.png'))))

class ApplyCLAHE(object):
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img):
        # Convert PIL Image to numpy array
        img_np = np.array(img)

        # Apply CLAHE to the green channel, as it represents the most clinically significant wavelength for lesion visualization.
        green_channel = img_np[:, :, 1]
        clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
        green_clahe = clahe.apply(green_channel)
        img_np[:, :, 1] = green_clahe

        # Convert back to PIL Image
        img_clahe_pil = Image.fromarray(img_np)

        return img_clahe_pil

In [ ]:

if config.compute_normalization: #automatic mean calculation and std calculation
        mean = np.mean([np.array(img).mean(axis=(0, 1)) for img, _ in train_dataset], axis=0) / 255.0
        std = np.std([np.array(img).std(axis=(0, 1)) for img, _ in train_dataset], axis=0) / 255.0
        print(f"Computed mean: {mean}, std: {std}")
else:
        mean = [0.485, 0.456, 0.406]
        std = [0.229, 0.224, 0.225]
        print(f"Using default mean: {mean}, std: {std}")

In [ ]:
def build_transforms(mean, std):
    train_transforms = transforms.Compose([
            ApplyCLAHE(clip_limit=2.0, tile_grid_size=(8, 8)),
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=mean, std=std)
        ])

    val_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=mean, std=std)
    ])

    return train_transforms, val_transforms

In [ ]:
class ImageFolderWithPaths(ImageFolder):
    """Custom dataset that includes image file paths. Extends torchvision.datasets.ImageFolder"""

    # override the __getitem__ method. this is the method that dataloader calls
    def __getitem__(self, index):
        # this is what ImageFolder normally returns 
        original_tuple = super(ImageFolderWithPaths, self).__getitem__(index)
        # the image file path
        path = self.imgs[index][0]
        # make a new tuple that includes original and the path
        tuple_with_path = (original_tuple + (path,))
        return tuple_with_path

In [ ]:
train_transforms, test_transforms = build_transforms(mean=config.mean, std=config.std)

In [ ]:
train_dataset = ImageFolder(root=os.path.join(config.chaksu_dir, 'train'), transform=train_transforms)
test_dataset = ImageFolder(root=os.path.join(config.chaksu_dir, 'test'), transform=test_transforms)
val_dataset = ImageFolder(root=os.path.join(config.chaksu_dir, 'val'), transform=test_transforms)  # Assuming you have a validation set

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)  # Using validation set for validation as well

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features
model.fc = nn.Sequential(
    nn.Linear(num_ftrs, 256),  # Assuming binary classification
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 2),  # Assuming binary classification

)
model = model.to(device)
print(model.fc) 

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=config.learning_rate, steps_per_epoch=len(train_loader), epochs=config.num_epochs)
focal_loss = FocalLoss(alpha=0.25, gamma=2.0, reduction='mean')  # Adjust alpha and gamma as needed
scaler = torch.amp.GradScaler('cuda')  # For mixed precision training and efficent memory usage

best_val_accuracy = 0.0
epoch_no_improvement = 0
accuracy = 0.0

start_epoch = 0

if os.path.exists(os.path.join(config.checkpoint_dir, 'checkpoint.pth')):
    checkpoint = torch.load(os.path.join(config.checkpoint_dir, 'checkpoint.pth'), map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_accuracy = checkpoint['best_val_accuracy']
    print(f"Resuming from epoch {start_epoch}")
else:
    print("No checkpoint found, creating a new one.")

for epoch in range(start_epoch, config.num_epochs):
    model.train()
    running_loss = 0.0
    for batch_idx, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            loss = focal_loss(outputs, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # Gradient clipping
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()
        if batch_idx % 10 == 0:  # Print every 10 batches
            print(f"Epoch [{epoch+1}/{config.num_epochs}], Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}")
    
    model.eval()
    with torch.no_grad():
        val_loss = 0.0
        correct = 0
        total = 0
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = focal_loss(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        val_loss /= len(val_loader)
        accuracy = correct / total
        print(f"Validation Loss: {val_loss:.4f}, Accuracy: {accuracy:.4f}")

    if accuracy > best_val_accuracy:
        epoch_no_improvement = 0
        best_val_accuracy = accuracy
        print(f"Saved best model with accuracy: {best_val_accuracy:.4f}")
        torch.save(model.state_dict(), os.path.join(config.checkpoint_dir, 'best_model.pth'))
    else:
        epoch_no_improvement += 1
        if epoch_no_improvement >= config.patience:
            print(f"No improvement for {config.patience} epochs. Stopping early at epoch {epoch+1}.")
            break

    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{config.num_epochs}] completed. Average Loss: {epoch_loss:.4f}")

    torch.save({
    'epoch':                epoch,
    'model_state_dict':     model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'best_val_accuracy':    best_val_accuracy,
    'val_accuracy':         accuracy,
}, os.path.join(config.checkpoint_dir, 'best_model.pth'))

In [ ]:
checkpoint = torch.load(os.path.join(config.checkpoint_dir, 'best_model.pth'), map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f"Loaded best model - Val Accuracy: {checkpoint['best_val_accuracy']:.4f}")

In [ ]:
def evaluation(model, test_loader, device, dataset_name, class_to_idx, output_dir):
    glaucoma_class_index = class_to_idx.get('glaucoma', None)
    if glaucoma_class_index is None:
        raise ValueError("Glaucoma class index not found in class_to_idx")

    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            probs = F.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    cm = confusion_matrix(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, output_dict=True)
    tn, fp, fn, tp = cm.ravel()

    results = {
        'confusion_matrix': cm.tolist(),
        'classification_report': report,
        'auc': roc_auc_score(all_labels, all_probs[:, glaucoma_class_index]),
        'sensitivity': tp / (tp + fn) if (tp + fn) > 0 else 0,
        'specificity': tn / (tn + fp) if (tn + fp) > 0 else 0,
        'kappa': cohen_kappa_score(all_labels, all_preds),
        'labels': all_labels.tolist(),
        'predictions': all_preds.tolist(),
        'probabilities': all_probs.tolist(),
        'dataset': dataset_name
    }
    
    with open(os.path.join(output_dir, f"{dataset_name}_results.json"), 'w') as f:
        json.dump(results, f, indent=4)

    print(f"\nEvaluation Results on {results['dataset']}: {results['dataset']}")
    print(f"Test AUC: {results['auc']:.4f}, Sensitivity: {results['sensitivity']:.4f}, Specificity: {results['specificity']:.4f}, Kappa: {results['kappa']:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    print(f"Classification Report:\n{json.dumps(report, indent=4)}")

    return results

chaksu_results = evaluation(model, test_loader, device, "Chaksu", train_dataset.class_to_idx, config.metrics_dir)

In [ ]:
domain_prefixes = ["Bosch", "Forus", "Remidio"]

device_indices = {domain: [] for domain in config.chaksu_devices}

for idx, (_, _, path) in enumerate(test_dataset.imgs):
    for domain in config.chaksu_devices:
        if domain in path:
            device_indices[domain].append(idx)
            break

In [ ]:
chaksu_full_train_tf = ImageFolderWithPaths(root=config.chaksu_dir, transform=train_transforms)
chaksu_full_test_tf = ImageFolderWithPaths(root=config.chaksu_dir, transform=test_transforms)

bosch_dataset = Subset(chaksu_full_test_tf, device_indices['Bosch'])
forus_dataset = Subset(chaksu_full_test_tf, device_indices['Forus'])
remido_dataset = Subset(chaksu_full_test_tf, device_indices['Remidio'])


bosch_train_idx, bosch_test_idx = train_test_split(device_indices['Bosch'], seed = config.seed, test_size=0.2)

bosch_train = Subset(chaksu_full_train_tf, bosch_train_idx)
bosch_val = Subset(chaksu_full_train_tf, bosch_test_idx)
bosch_train_loader = DataLoader(bosch_train, batch_size=config.batch_size, shuffle=True, num_workers=2, pin_memory=True)
bosch_test_loader = DataLoader(bosch_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2, pin_memory=True)

forus_loader = DataLoader(forus_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2, pin_memory=True)
remidio_loader = DataLoader(remido_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2, pin_memory=True)

model_device_results = build_model

In [ ]:
domain_results = {}
for domain in config.chaksu_devices:
    domain_dataset = ImageFolder(root=os.path.join(config.chaksu_dir, domain), transform=test_transforms)
    domain_loader = DataLoader(domain_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    domain_results[domain] = evaluation(model, domain_loader, device, f"{domain_prefixes[domain]} (Zero-Shot)", train_dataset.class_to_idx, config.metrics_dir)

In [ ]:
arcmia_dataset = ImageFolder(root=os.path.join(config.data_dir, 'arcmia'), transform=test_transforms)
arcmia_loader = DataLoader(arcmia_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
arcmia_results = evaluation(model, arcmia_loader, device, "ARCMIA", train_dataset.class_to_idx, config.output_dir)

In [ ]:
orgia_dataset = ImageFolder(root=os.path.join(config.data_dir, 'orgia'), transform=test_transforms)
orgia_loader = DataLoader(orgia_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
orgia_results = evaluation(model, orgia_loader, device, "ORGIA", train_dataset.class_to_idx, config.output_dir)

In [ ]:
refuge_dataset = ImageFolder(root=os.path.join(config.data_dir, 'refuge'), transform=test_transforms)
refuge_loader = DataLoader(refuge_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
refuge_results = evaluation(model, refuge_loader, device, "REFUGE", train_dataset.class_to_idx, config.output_dir)

In [ ]:
all_results = {
    "Chaksu (Domain)": chaksu_results,
    "ARCMIA (Zero-Shot)": arcmia_results,
    "ORGIA (Zero-Shot)": orgia_results,
    "REFUGE (Zero-Shot)": refuge_results
}

baseline = chaksu_results # Assuming Chaksu is the baseline for comparison  
rows = []
for name in all_results.keys():
    results = all_results[name]
    row = {
        "Dataset": name,
        "AUC": results['auc'],
        "Sensitivity": results['sensitivity'],
        "Specificity": results['specificity'],
        "Kappa": results['kappa'],
        "AUC Difference": results['auc'] - baseline['auc'],
        "Sensitivity Difference": results['sensitivity'] - baseline['sensitivity'],
        "Specificity Difference": results['specificity'] - baseline['specificity'],
    }
    rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.to_csv(os.path.join(config.output_dir, "summary_results.csv"), index=False)
print("\nSummary of Results:")
print(summary_df)

In [ ]:
def fairness_metrics(all_results, baseline_results):
    fairness_metrics = {}
    for dataset_name, results in all_results.items():
        fairness_metrics[dataset_name] = {
            "True Positive Rate": results['sensitivity'],
            "False Positive Rate": 1 - results['specificity'],
            "Accuracy": results['classification_report']['accuracy'],
            "AUC Difference": results['auc'] - baseline_results['auc'],
            "Sensitivity Difference": results['sensitivity'] - baseline_results['sensitivity'],
            "Specificity Difference": results['specificity'] - baseline_results['specificity'],
        }
    return fairness_metrics

def pairwise_tpr_gaps(fairness_metrics):
    names = list(fairness_metrics.keys())
    gaps = {}
    for a, b in itertools.combinations(names, 2):
        gap = abs(fairness_metrics[a]["True Positive Rate"] - fairness_metrics[b]["True Positive Rate"])
        gaps[(a, b)] = gap
    return gaps

def pairwise_fpr_gaps(fairness_metrics):
    names = list(fairness_metrics.keys())
    gaps = {}
    for a, b in itertools.combinations(names, 2):
        gap = abs(fairness_metrics[a]["False Positive Rate"] - fairness_metrics[b]["False Positive Rate"])
        gaps[(a, b)] = gap
    return gaps

fairness_results = fairness_metrics(all_results, chaksu_results)
tpr_gaps = pairwise_tpr_gaps(fairness_results)
fpr_gaps = pairwise_fpr_gaps(fairness_results)

equal_opportunity_difference = max(tpr_gaps.values())
equalized_odds_gap = max(max(tpr_gaps.values()), max(fpr_gaps.values()))

accuracies = {name: fairness_results[name]["Accuracy"] for name in fairness_results}
worst_group = min(accuracies, key=accuracies.get)
worst_group_accuracy = accuracies[worst_group]

In [ ]:
cam = HiResCAM(model=model, target_layers=[model.layer4[-1]], use_cuda=torch.cuda.is_available())

In [ ]:
def generate_cam_visualizations(model, cam, dataset, class_to_idx, device, output_dir, dataset_name, num_samples=5):
    model.eval()
    os.makedirs(output_dir, exist_ok=True)

    glaucoma_idx = class_to_idx.get('glaucoma', None)
    if glaucoma_idx is None:
        raise ValueError("Glaucoma class index not found in class_to_idx")

    sample_indices = np.random.choice(len(dataset), num_samples, replace=False)

    for idx in sample_indices:
        img, label = dataset[idx]
        input_tensor = img.unsqueeze(0).to(device)

        targets = [ClassifierOutputTarget(glaucoma_idx)]
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
        grayscale_cam = grayscale_cam[0, :]

        img_np = img.permute(1, 2, 0).cpu().numpy()
        img_np = (img_np * np.array(config.std) + np.array(config.mean)) * 255.0
        img_np = np.clip(img_np, 0, 255).astype(np.uint8)

        heatmap = cv2.applyColorMap(np.uint8(255 * grayscale_cam), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

        superimposed_img = heatmap.astype(np.float32) * 0.4 + img_np.astype(np.float32)
        superimposed_img = np.clip(superimposed_img, 0, 255).astype(np.uint8)

        plt.figure(figsize=(10, 5))
        plt.subplot(1, 3, 1)
        plt.imshow(img_np)
        plt.title(f"Original\nLabel: {label}")
        plt.axis('off')

        plt.subplot(1, 3, 2)
        plt.imshow(grayscale_cam, cmap='jet')
        plt.title("CAM Heatmap")
        plt.axis('off')

        plt.subplot(1, 3, 3)
        plt.imshow(superimposed_img)
        plt.title("Superimposed")
        plt.axis('off')

        plt.suptitle(dataset_name)
        output_path = os.path.join(output_dir, f"{dataset_name}_cam_{idx}.png")
        plt.savefig(output_path, bbox_inches='tight')
        plt.close()

In [ ]:
generate_cam_visualizations(model, cam, test_dataset, train_dataset.class_to_idx, device, config.output_dir, "Chaksu")
generate_cam_visualizations(model, cam, arcmia_dataset, train_dataset.class_to_idx, device, config.output_dir, "ARCMIA")
generate_cam_visualizations(model, cam, orgia_dataset, train_dataset.class_to_idx, device, config.output_dir, "ORGIA")
generate_cam_visualizations(model, cam, refuge_dataset, train_dataset.class_to_idx, device, config.output_dir, "REFUGE")

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, PageBreak,
                                 Table, TableStyle, Image as RLImage)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib import colors
import glob


def generate_iris_report(all_results, fairness_results, tpr_gaps, fpr_gaps,
                          equal_opportunity_difference, equalized_odds_gap,
                          worst_group, worst_group_accuracy,
                          summary_df, config, report_name="IRIS_report.pdf"):
    """
    Builds a single PDF pulling together cross-domain metrics, fairness metrics,
    and CAM visualizations already saved to disk. Reads only from objects already
    in memory (all_results, fairness_results, summary_df) and images already
    written to config.cam_dir -- does not re-run inference.
    """
    report_path = os.path.join(config.reports_dir, report_name)
    doc = SimpleDocTemplate(report_path, pagesize=letter,
                             topMargin=0.7*inch, bottomMargin=0.7*inch)
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name="Caption", fontSize=9, textColor=colors.grey))
    story = []

    # --- Title page ---
    story.append(Paragraph("IRIS: Cross-Domain Glaucoma Detection Report", styles['Title']))
    story.append(Paragraph(f"Trained on Chákṣu, evaluated zero-shot on ACRIMA, ORIGA, REFUGE-2020",
                            styles['Normal']))
    story.append(Paragraph(f"Generated: {time.strftime('%Y-%m-%d %H:%M')}", styles['Caption']))
    story.append(Spacer(1, 24))

    # --- Section 1: Cross-domain performance table ---
    story.append(Paragraph("1. Cross-Domain Performance", styles['Heading1']))
    table_data = [["Dataset", "AUC", "Sensitivity", "Specificity", "Kappa"]]
    for _, row in summary_df.iterrows():
        table_data.append([
            row["Dataset"],
            f"{row['AUC']:.3f}",
            f"{row['Sensitivity']:.3f}",
            f"{row['Specificity']:.3f}",
            f"{row.get('Kappa', float('nan')):.3f}" if 'Kappa' in row else "-",
        ])
    perf_table = Table(table_data, hAlign='LEFT')
    perf_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#2c3e50")),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('FONTSIZE', (0, 0), (-1, -1), 9),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor("#f2f2f2")]),
        ('TOPPADDING', (0, 0), (-1, -1), 4),
        ('BOTTOMPADDING', (0, 0), (-1, -1), 4),
    ]))
    story.append(perf_table)
    story.append(Spacer(1, 6))
    story.append(Paragraph(
        "Chaksu (in-domain) serves as the baseline; ACRIMA/ORIGA/REFUGE reflect zero-shot "
        "generalization to Western clinical populations.", styles['Caption']))
    story.append(Spacer(1, 20))

    # --- Section 2: Confusion matrices ---
    story.append(Paragraph("2. Confusion Matrices", styles['Heading1']))
    for name, res in all_results.items():
        cm = np.array(res['confusion_matrix'])
        fig, ax = plt.subplots(figsize=(3, 2.6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
        ax.set_title(name, fontsize=10)
        ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
        cm_path = os.path.join(config.figures_dir, f"cm_{name.replace(' ', '_').replace('(', '').replace(')', '')}.png")
        fig.tight_layout()
        fig.savefig(cm_path, dpi=150)
        plt.close(fig)
        story.append(RLImage(cm_path, width=2.8*inch, height=2.4*inch))
    story.append(PageBreak())

    # --- Section 3: Fairness metrics ---
    story.append(Paragraph("3. Fairness Metrics", styles['Heading1']))
    story.append(Paragraph(
        f"<b>Equal Opportunity Difference:</b> {equal_opportunity_difference:.4f} "
        f"(largest pairwise TPR gap across the 4 populations)", styles['Normal']))
    story.append(Paragraph(
        f"<b>Equalized Odds Gap:</b> {equalized_odds_gap:.4f} "
        f"(worse of the largest TPR gap and largest FPR gap)", styles['Normal']))
    story.append(Paragraph(
        f"<b>Worst-Group Accuracy:</b> {worst_group_accuracy:.4f} (group: {worst_group})",
        styles['Normal']))
    story.append(Spacer(1, 10))

    fair_table_data = [["Dataset", "TPR", "FPR", "Accuracy"]]
    for name, vals in fairness_results.items():
        fair_table_data.append([
            name, f"{vals['True Positive Rate']:.3f}",
            f"{vals['False Positive Rate']:.3f}",
            f"{vals['Accuracy']:.3f}",
        ])
    fair_table = Table(fair_table_data, hAlign='LEFT')
    fair_table.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#2c3e50")),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('FONTSIZE', (0, 0), (-1, -1), 9),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.white, colors.HexColor("#f2f2f2")]),
    ]))
    story.append(fair_table)
    story.append(Spacer(1, 14))

    # Pairwise TPR gap heatmap
    names = list(fairness_results.keys())
    gap_matrix = pd.DataFrame(0.0, index=names, columns=names)
    for (a, b), gap in tpr_gaps.items():
        gap_matrix.loc[a, b] = gap
        gap_matrix.loc[b, a] = gap
    fig, ax = plt.subplots(figsize=(4.5, 3.8))
    sns.heatmap(gap_matrix, annot=True, fmt=".3f", cmap='Reds', ax=ax)
    ax.set_title("Pairwise TPR Gap (Equal Opportunity)", fontsize=10)
    plt.xticks(rotation=30, ha='right', fontsize=7)
    plt.yticks(fontsize=7)
    gap_path = os.path.join(config.figures_dir, "tpr_gap_heatmap.png")
    fig.tight_layout()
    fig.savefig(gap_path, dpi=150)
    plt.close(fig)
    story.append(RLImage(gap_path, width=4.2*inch, height=3.5*inch))
    story.append(PageBreak())

    # --- Section 4: Grad-CAM / HiResCAM visualizations ---
    story.append(Paragraph("4. HiResCAM Attention Visualizations", styles['Heading1']))
    story.append(Paragraph(
        "Localization consistency across domains -- checking whether the model "
        "attends to the optic disc/cup region regardless of source population.",
        styles['Caption']))
    story.append(Spacer(1, 8))

    cam_images = sorted(glob.glob(os.path.join(config.cam_dir, "*.png")))
    if not cam_images:
        story.append(Paragraph("No CAM images found in config.cam_dir.", styles['Normal']))
    for img_path in cam_images:
        story.append(RLImage(img_path, width=5.5*inch, height=2.75*inch))
        story.append(Paragraph(os.path.basename(img_path), styles['Caption']))
        story.append(Spacer(1, 6))

    doc.build(story)
    print(f"Report saved to: {report_path}")
    return report_path


report_path = generate_iris_report(
    all_results=all_results,
    fairness_results=fairness_results,
    tpr_gaps=tpr_gaps,
    fpr_gaps=fpr_gaps,
    equal_opportunity_difference=equal_opportunity_difference,
    equalized_odds_gap=equalized_odds_gap,
    worst_group=worst_group,
    worst_group_accuracy=worst_group_accuracy,
    summary_df=summary_df,
    config=config,
)
